# Columnar analysis

## Introduction

"Columnar" is a term that tends to get overloaded, and I’ve used it in two distinct ways:

* **Data layout:** organizing data in memory or on disk to enable faster, selective readout.
* **Array-oriented computation:** performing operations directly on entire arrays of data, rather than processing one value at a time in a loop. In other words — _no for loops_!

Of these, **only the second meaning directly affects physicists writing analysis code**.

That’s why it might make more sense calling it **"array-oriented programming"** — it’s a programming paradigm, much like "imperative" or "functional," describing _how you think about and organize your code_.

### Easy example of imperative, functional, and array-oriented

Compute the square of every element in a list/array.

#### Imperative

In [ ]:
original = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

result = []
for x in original:
    result.append(x**2)

result

#### Functional

In [ ]:
original = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

result = list(map(lambda x: x**2, original))

result

Functional programming with `map` isn't common in Python, but list comprehensions are pretty close to the "spirit" of functional programming:

In [ ]:
original = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

result = [x**2 for x in original]

result

#### Array-oriented

In [ ]:
import numpy as np

In [ ]:
original = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

result = original**2

result

### Hard example of imperative, functional, and array-oriented

Compute gravitational forces among $n$ particles in 3 dimensions.

#### Newton’s Law of Universal Gravitation

The gravitational force between two point masses $m_1$ and $m_2$ separated by a distance $r$ is given by:

$$
\vec{F} = G \frac{m_1 m_2}{r^2} \hat{r}
$$

Where:

* $\vec{F}$ is the gravitational force vector (Newtons)
* $G$ is the gravitational constant
* $m_1, m_2$ are the masses of the two objects
* $r$ is the distance between their centers
* $\hat{r}$ is the unit vector pointing from one mass to the other

In component form for positions $\vec{x}_i$ and $\vec{x}_j$:

$$
\vec{F}_{ij} = G \frac{m_i m_j}{|\vec{r}_{ij}|^3} \vec{r}_{ij}
$$

with

$$
\vec{r}_{ij} = \vec{x}_j - \vec{x}_i
$$


<br><br><br>

In [ ]:
m = np.array([100, 1, 1])  # sun and a double-planet (a 3-body problem)

# initial position (x) and momentum (p)
x = np.array([[0, 0, 0], [0, 0.9, 0], [0, 1.1, 0]])
p = np.array([[0, 0, 0], [-13, 0, 0], [-10, 0, 0]])

G = 1

#### Imperative

In [ ]:
def imperative_forces(m, x, p):
    # Initialize total force array with zeros, same shape as positions array x
    total_force = np.zeros_like(x)

    # Loop over each unique pair of particles (i, j) where i < j
    for i in range(len(x)):
        for j in range(i + 1, len(x)):
            # Extract masses of the two particles
            mi, mj = m[i], m[j]

            # Extract positions of the two particles
            xi, xj = x[i], x[j]

            # Compute displacement vector from particle i to particle j
            displacement = [
                xj[0] - xi[0],
                xj[1] - xi[1],
                xj[2] - xi[2],
            ]

            # Compute Euclidean distance between the two particles
            distance = np.sqrt(
                displacement[0] ** 2 + displacement[1] ** 2 + displacement[2] ** 2
            )

            # Compute unit vector (direction) from particle i to particle j
            direction = [
                displacement[0] / distance,
                displacement[1] / distance,
                displacement[2] / distance,
            ]

            # Calculate gravitational force magnitude and vector components
            # Newton's law of gravitation: F = G * m_i * m_j / r^2 * direction
            force = [
                G * mi * mj * direction[0] / distance**2,
                G * mi * mj * direction[1] / distance**2,
                G * mi * mj * direction[2] / distance**2,
            ]

            # Add the force to particle i (towards particle j)
            total_force[i, 0] += force[0]
            total_force[i, 1] += force[1]
            total_force[i, 2] += force[2]

            # Subtract the force from particle j (equal and opposite reaction)
            total_force[j, 0] += -force[0]
            total_force[j, 1] += -force[1]
            total_force[j, 2] += -force[2]

    # Return the array of total forces on each particle
    return total_force

#### Functional

In [ ]:
from functools import reduce
from itertools import combinations

In [ ]:
def functional_forces(m, x, p):
    def negate(vector):
        # Negate each component of the vector
        return [-a for a in vector]

    def add(*vectors):
        # Component-wise sum of multiple vectors
        return [reduce(lambda a, b: a + b, components) for components in zip(*vectors)]

    def subtract(vectorA, vectorB):
        # Vector subtraction: vectorA - vectorB
        return add(vectorA, negate(vectorB))

    def magnitude(vector):
        # Euclidean norm of the vector
        return np.sqrt(reduce(lambda a, b: a + b, map(lambda a: a**2, vector)))

    def force(mi, mj, xi, xj, pi, pj):
        # Calculate gravitational force vector exerted by j on i
        displacement = subtract(xj, xi)  # Note: corrected displacement to point i->j
        distance = magnitude(displacement)
        direction = [a / distance for a in displacement]
        return [G * mi * mj * a / distance**2 for a in direction]

    # Generate all unique pairs of particles and their forces
    pairwise_forces = [
        ((i, j), force(mi, mj, xi, xj, pi, pj))
        for ((i, (mi, xi, pi)), (j, (mj, xj, pj))) in combinations(
            enumerate(zip(m, x, p)), 2
        )
    ]

    def partial_forces(pairwise_forces, i):
        # Collect all forces acting on particle i
        return (
            [force for ((a, b), force) in pairwise_forces if a == i]  # force from j->i
            + [
                negate(force) for ((a, b), force) in pairwise_forces if b == i
            ]  # equal and opposite force from i->j
        )

    # Sum all partial forces for each particle and return as numpy array
    return np.array([add(*partial_forces(pairwise_forces, i)) for i in range(len(m))])

#### Array-oriented

In [ ]:
def array_forces(m, x, p):
    # Get the indices of the upper triangle of the pairwise interaction matrix (i < j)
    # This ensures each pair is considered only once (no double counting)
    i, j = np.triu_indices(len(x), k=1)

    # Calculate displacement vectors between pairs of particles (j - i)
    pw_displacement = x[j] - x[i]

    # Compute the Euclidean distance between particle pairs
    # Sum of squared components followed by square root gives distance
    pw_distance = np.sqrt(np.sum(pw_displacement**2, axis=-1))

    # Calculate the unit direction vectors from particle i to particle j
    # Divide displacement vector by distance to normalize
    pw_direction = pw_displacement / pw_distance[:, np.newaxis]

    # Calculate the gravitational force vector for each pair using Newton's law of gravitation:
    # F = G * m_i * m_j / r^2 * direction
    # The [:, np.newaxis] adds a new axis for broadcasting over x,y,z components
    pw_force = (
        G
        * m[i, np.newaxis]
        * m[j, np.newaxis]
        * pw_direction
        / pw_distance[:, np.newaxis] ** 2
    )

    # Initialize array to accumulate total forces on each particle (same shape as x)
    total_force = np.zeros_like(x)

    # Add force vectors to particle i's total force
    np.add.at(total_force, i, pw_force)

    # Subtract equal and opposite force vectors from particle j's total force
    # This enforces Newton's third law: action = -reaction
    np.add.at(total_force, j, -pw_force)

    # Return the total forces acting on all particles
    return total_force

In [ ]:
imperative_forces(m, x, p)

In [ ]:
functional_forces(m, x, p)

In [ ]:
array_forces(m, x, p)

#### Let's see it!

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

In [ ]:
def array_step(m, x, p, dt):
    # this is a numerically stable way of updating positions, momenta, and forces
    p += array_forces(m, x, p) * (dt / 2)  # half kick
    x += p * dt / m[:, np.newaxis]  # full drift
    p += array_forces(m, x, p) * (dt / 2)  # half kick

In [ ]:
def plot(m, x, p, dt, num_frames=100, steps_per_frame=10):
    num_particles = len(m)  # Number of particles

    # Allocate an empty array to store particle positions over time
    # Shape: (num_frames, num_particles, 2) for 2D positions (x, y)
    history = np.empty((num_frames, num_particles, 2))

    # Simulate and record positions at each frame
    for i in range(num_frames):
        # Save current x positions for all particles
        history[i, :, 0] = x[:, 0]
        # Save current y positions for all particles
        history[i, :, 1] = x[:, 1]

        # Advance simulation by 'steps_per_frame' small steps between frames
        for _ in range(steps_per_frame):
            array_step(m, x, p, dt)  # Update positions and momenta in place

    # Set up the plotting figure and axis with fixed size
    fig, ax = plt.subplots(figsize=(5, 5))

    lines = []
    # Initialize empty trajectory lines for each particle
    for j in range(num_particles):
        # Plot initial line segment (single point for now) for each particle
        (line,) = ax.plot(history[:1, j, 0], history[:1, j, 1])
        lines.append(line)

    # Plot particles as scatter points at initial positions
    dots = ax.scatter(history[0, :, 0], history[0, :, 1])

    # Fix axis limits to keep view constant during animation
    ax.set_xlim(-2, 2)
    ax.set_ylim(-2, 2)

    # Define update function called at each animation frame
    def update(i):
        # Update trajectory lines to show all positions up to frame i
        for j, line in enumerate(lines):
            line.set_xdata(history[:i, j, 0])
            line.set_ydata(history[:i, j, 1])
        # Update scatter plot positions to frame i positions
        dots.set_offsets(history[i, :, :])
        # Return updated artists for blitting (efficient redraw)
        return [*lines, dots]

    # Create animation object
    ani = animation.FuncAnimation(
        fig=fig,  # Figure to animate
        func=update,  # Update function called each frame
        frames=num_frames,  # Total number of frames
        interval=50,  # Delay between frames in milliseconds
        blit=True,  # Use blitting for performance
    )

    # Convert animation to HTML5 format for display (e.g., in Jupyter)
    out = HTML(ani.to_jshtml())

    # Close figure to avoid duplicate static plot display
    plt.close()

    # Return HTML animation object for display
    return out

In [ ]:
m = np.array([100, 1, 1], np.float64)
x = np.array([[0, 0, 0], [0, 0.9, 0], [0, 1.1, 0]], np.float64)
p = np.array([[0, 0, 0], [-13, 0, 0], [-10, 0, 0]], np.float64)

plot(m, x, p, dt=0.001)

In [ ]:
# Define momentum components (velocity * mass) for particles 0 and 1
a = 0.347111
b = 0.532728

# Mass array for 3 particles, all unit mass
m = np.array([1, 1, 1], np.float64)

# Initial positions of particles in 3D space
# Particle 0 at (-1, 0, 0)
# Particle 1 at (1, 0, 0)
# Particle 2 at origin (0, 0, 0)
x = np.array([[-1, 0, 0], [1, 0, 0], [0, 0, 0]], np.float64)

# Initial momenta of particles (momentum = mass * velocity)
# Particles 0 and 1 have the same momentum vector (a, b, 0)
# Particle 2 has momentum exactly opposite to the sum of the other two,
# ensuring total momentum sums to zero (momentum conservation)
p = np.array(
    [
        [a, b, 0],  # Particle 0
        [a, b, 0],  # Particle 1
        [-2 * a, -2 * b, 0],  # Particle 2
    ],
    np.float64,
)

# Run simulation and plot particle trajectories with timestep dt=0.01
plot(m, x, p, dt=0.01)

In [ ]:
# Create an array of masses for 25 particles, all with mass = 1
m = np.ones(25)

# Initialize positions of 25 particles randomly in 3D space
# Each coordinate is drawn from a normal distribution with mean=0 and std=1
x = np.random.normal(0, 1, (25, 3))

# Initialize momenta of 25 particles randomly in 3D
# Each component drawn from normal distribution with mean=0 and std=1
p = np.random.normal(0, 1, (25, 3))

# Run simulation and animate particle trajectories with a timestep of 0.0025
plot(m, x, p, dt=0.0025)

#### Let's time it!

In [ ]:
m = np.ones(500)
x = np.random.normal(0, 1, (500, 3))
p = np.random.normal(0, 1, (500, 3))

In [ ]:
%%timeit -n1 -r1

imperative_forces(m, x, p)

In [ ]:
%%timeit -n1 -r1

functional_forces(m, x, p)

In [ ]:
%%timeit -n1 -r1

array_forces(m, x, p)

In [ ]:
import dis

disassembled = dis.Bytecode(imperative_forces)
instructions = list(disassembled)
print(f"Number of bytecode instructions: {len(instructions)}")

In [ ]:
disassembled = dis.Bytecode(functional_forces)
instructions = list(disassembled)
print(f"Number of bytecode instructions: {len(instructions)}")

In [ ]:
disassembled = dis.Bytecode(array_forces)
instructions = list(disassembled)
print(f"Number of bytecode instructions: {len(instructions)}")

In Python, array-oriented programming is a big advantage because it avoids Python's overhead (virtual machine, dynamic data types, garbage collection, etc).

* In `imperative_forces`, each of the 500 × 499 pairs of particles has to step through 30 lines of Python code (234 byte-code instructions).
* In `functional_forces`, each of the 500 × 499 pairs of particles has to step through functions that add up to 24 lines of Python code (137 byte-code instructions), plus call-stack overhead.
* `array_forces` has 10 lines of Python code (88 byte-code instructions), but the Python virtual machine steps through them only once, _not once per pair of particles_.

This is also relevant for GPU programming.

To get the most performance out of GPU programming frameworks like CUDA, you need to arrange the computation as array-at-a-time and think about vectorization.

### What if imperative code is easier to reason about?

Sometimes it is.

If it's easier to think about a problem imperatively, but the loop would iterate over some large number, just make sure Python isn't implementing the loop.

* Just-In-Time (JIT) compile it with [Numba](https://numba.pydata.org/).
* Use ROOT's [RDataFrame](https://root.cern/doc/master/classROOT_1_1RDataFrame.html) to compile and run C++ over ROOT data.
* Use [cppyy](https://cppyy.readthedocs.io/) to compile and run C++ over arbitrary data.
* Use [Julia](https://julialang.org/).
* Use [pybind11](https://pybind11.readthedocs.io/) to compile a Python extension in C++ or [PyO3](https://pyo3.rs/) to compile a Python extension in Rust.

All of these involve more set-up time to get started than array-oriented programming, but may be easier to deal with in the long run, depending on the problem.

## Regular multi-dimensional arrays

In a **3D NumPy array**, the `axis` parameter specifies **which dimension an operation acts along**.

:::{figure} ./images/numpy_3d_axis.png
:align: center
:width: 50%
:::

Since a 3D array has three dimensions, `axis` can take one of three possible values:

* `axis=0` → operates along the <b>first dimension</b>, moving <b>between 2D arrays</b> stacked along the "depth" of the 3D structure.
* `axis=1` → operates along the <b>second dimension</b>, moving <b>across rows within each 2D array</b> — like the "height" within a layer.
* `axis=2` → operates along the <b>third dimension</b>, moving <b>across columns within each row</b> — representing the "width" of the 3D array.

In short:

* `axis=0` → across different 2D slices
* `axis=1` → down rows in each 2D slice
* `axis=2` → across columns in each row

This `axis` argument lets you control the direction of operations like `sum()`, `mean()`, or `max()` in multi-dimensional arrays.

## Puzzles

### NumPy puzzle 1

Given a 3D array,

In [ ]:
import numpy as np

In [ ]:
array3d = np.arange(2 * 3 * 5).reshape(2, 3, 5)
array3d

you can select items

:::{figure} ./images/array3d-highlight1.svg
:align: center
:width: 25%
:::

with

In [ ]:
array3d[:, 1:, 1:]

Write a slice the selects these elements:

:::{figure} ./images/array3d-highlight2.svg
:align: center
:width: 25%
:::

In [ ]:
# Write your code here


:::{note} Solution (Don't look until you're done)
:class: dropdown

Either keep the number of dimensions intact (the first dimension has length 1):
```python
array3d[:1, :, 2:]
```
or reduce that dimension:
```python
array3d[0, :, 2:]
```
These are different answers to the question, but both are fine.
:::

### NumPy puzzle 2

Compute the size of the spaces between consecutive elements in the following array.

In [ ]:
array = np.array([1.1, 2.2, 3.3, 4.4, 5.5, 6.6, 7.7, 8.8, 9.9])
array

**Hint:**

:::{figure} ./images/flat-operation.svg
:align: center
:width: 40%
:::
:::{figure} ./images/shifted-operation.svg
:align: center
:width: 40%
:::

In [ ]:
# Write your code here


:::{note} Solution (Don't look until you're done)
:class: dropdown

This involves either slicing and applying a universal function across all elements, showing how NumPy's elementary operations can be combined to get functionality that may not be explicitly compiled into the library or using a NumPy's diff function.

```python
array[1:] - array[:-1]
```
or
```python
np.diff(array)
```
:::

### NumPy puzzle 3

Compute the length of this curve.

:::{figure} ./images/length-by-segment.svg
:align: center
:width: 50%
:::

In [ ]:
t = np.linspace(0, 2 * np.pi, 10000)
x = np.sin(3 * t)
y = np.sin(4 * t)

In [ ]:
plt.plot(x, y);

In [ ]:
# Write your solution here


:::{note} Solution (Don't look until you're done)
:class: dropdown

This calculation of path length involves slicing, universal functions, and reduction.
```python
np.sum(np.sqrt((x[1:] - x[:-1])**2 + (y[1:] - y[:-1])**2), axis=0)
```

### NumPy puzzle 4

Scale this image down by a factor of 64 on both sides, using only [np.reshape](https://numpy.org/doc/stable/reference/generated/numpy.reshape.html), [np.mean](https://numpy.org/doc/stable/reference/generated/numpy.mean.html), and [np.ndarray.astype](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.astype.html).

In [ ]:
import matplotlib.image

In [ ]:
image = matplotlib.image.imread("images/sun-shines-in-CMS.jpg")
plt.imshow(image);

The current shape is

In [ ]:
image.shape

1920 rows, 2560 columns, and the third axis is for (red, green, blue), all `np.uint8`.

Your strategy should be to reshape the array, such that the dimension of length `1920` becomes two new dimensions of length `1920 // 64` and `64` and the dimension of length `2560` becomes two new dimensions of length `2560 // 64` and `64`. Then average over each of the dimensions of length `64`.

The shape should change as

$$\left(1920, 2560, 3\right) \to \left(\frac{1920}{64}, 64, \frac{2560}{64}, 64, 3\right) \to \left(\frac{1920}{64}, \frac{2560}{64}, 3\right)$$

and then you need to turn the floating-point dtype back into unsigned 8-bit integers with [np.ndarray.astype](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.astype.html).

In [ ]:
# Write your code here


:::{note} Solution (Don't look until you're done)
:class: dropdown

In NumPy, the shapes of arrays are fluid. An array can be reshaped from 3 dimensional to 5 dimensional as long as the number of items is unchanged.

Reshaping provides new axes for reducers like [np.mean](https://numpy.org/doc/stable/reference/generated/numpy.mean.html) to apply.

Be sure to average over axis 3 before axis 1, since reduction removes an axis, changing the numbering of later axes.

```python
resampled = np.mean(
    np.mean(
        image.reshape(1920 // 64, 64, 2560 // 64, 64, 3),
        axis=3,
    ),
    axis=1,
)
```
Or alternatively, use negative axis numbers and do it in the other order.
```python
resampled = np.mean(
    np.mean(
        image.reshape(1920 // 64, 64, 2560 // 64, 64, 3),
        axis=-4,
    ),
    axis=-2,
)
```
Or as a third option, use `keepdims=True` to keep the axis numbering from changing and only remove the length 1 dimensions after all reductions are done.
```python
resampled = np.mean(
    np.mean(
        image.reshape(1920 // 64, 64, 2560 // 64, 64, 3),
        axis=1,
        keepdims=True,
    ),
    axis=3,
    keepdims=True,
).reshape(1920 // 64, 2560 // 64, 3)
```
And then we can show it with
```python
plt.imshow(resampled.astype(np.uint8));
```
:::